# Implement Optimizers from Scratch (Adam / AdamW / Muon) - SOLUTION

**Difficulty**: 🔴 Hard

**Companies**: Meta, Google

---

### Problem Statement

"Write Adam from scratch" is a classic ML interview ask — it tests whether you know what an optimizer actually *does* between `loss.backward()` and `opt.step()`. You will implement three optimizers that together span a decade of the field, all following PyTorch's `torch.optim.Optimizer` interface:

1. **Adam** — Adaptive Moment Estimation (Kingma & Ba, 2014): momentum plus per-parameter adaptive learning rates.
2. **AdamW** — Adam with *decoupled* weight decay (Loshchilov & Hutter, 2017): the default optimizer for training transformers.
3. **Muon** — Momentum Orthogonalized by Newton–Schulz (Jordan et al., 2024): the optimizer behind recent LLM training speedrun records.

### Tasks

1. `newton_schulz(X, steps)` — approximate matrix orthogonalization (the helper Muon needs).
2. `MyAdam` — the Adam update rule, including bias correction.
3. `MyAdamW` — Adam with weight decay applied **directly to the parameters**, not through the gradient.
4. `MyMuon` — momentum followed by orthogonalization for matrix parameters; an AdamW fallback for 1D parameters.

### References

- Adam paper: https://arxiv.org/abs/1412.6980
- AdamW paper: https://arxiv.org/abs/1711.05101
- Muon blog: https://kellerjordan.github.io/posts/muon/


In [ ]:
import math
import torch
import torch.nn as nn
from torch.optim import Optimizer


## Helper: Newton–Schulz Orthogonalization

Muon needs to turn an arbitrary update matrix into an approximately **orthogonal** one — same shape, but with `O @ O.T ≈ I` (for a wide matrix). SVD would do it exactly but is too slow inside a training loop; Muon uses the quintic **Newton–Schulz iteration** instead:

```
X_{k+1} = a·X_k + b·(X_k X_kᵀ) X_k + c·(X_k X_kᵀ)² X_k      a = 15/8,  b = −5/4,  c = 3/8
```

The iteration only converges when the spectral norm of the input is at most 1, so normalize first — the Frobenius norm is a cheap upper bound on the spectral norm.

**Tip:** for an `(m, n)` matrix, `X @ X.T` is `(m, m)` and `X.T @ X` is `(n, n)` — build whichever Gram matrix is *smaller*.


In [ ]:
def newton_schulz(X: torch.Tensor, steps: int = 5) -> torch.Tensor:
    """
    Approximate orthogonalization of a matrix via Newton–Schulz iteration.

    Given X (m × n), return an approximately orthogonal matrix O of the same
    shape: O @ O.T ≈ I if m ≤ n, or O.T @ O ≈ I if m > n.

    Uses the quintic (5th-order) iteration with coefficients
    a = 15/8, b = -5/4, c = 3/8.

    Args:
        X:     input matrix, shape (m, n)
        steps: number of Newton–Schulz iterations (default 5)

    Returns:
        Orthogonalized matrix, same shape as X
    """
    # Normalize so the spectral norm is <= 1 (Frobenius norm >= spectral norm)
    X = X / (X.norm() + 1e-10)

    a, b, c = 15.0 / 8.0, -5.0 / 4.0, 3.0 / 8.0

    for _ in range(steps):
        if X.shape[0] <= X.shape[1]:
            # m <= n: work with the (m, m) Gram matrix
            A = X @ X.T
            X = a * X + b * (A @ X) + c * (A @ A @ X)
        else:
            # m > n: work with the (n, n) Gram matrix (smaller)
            A = X.T @ X
            X = a * X + b * (X @ A) + c * (X @ A @ A)

    return X


## Part 1: Adam

Adam combines two ideas:

- **Momentum** — an exponential moving average of past gradients.
- **RMSProp** — a per-parameter learning rate from a moving average of squared gradients.

```
m_t = β₁·m_{t−1} + (1−β₁)·g_t        (biased first moment)
v_t = β₂·v_{t−1} + (1−β₂)·g_t²       (biased second moment)
m̂_t = m_t / (1 − β₁ᵗ)                (bias corrections)
v̂_t = v_t / (1 − β₂ᵗ)
θ_t = θ_{t−1} − lr·m̂_t / (√v̂_t + ε)
```

**Key detail:** `β₁ᵗ` is β₁ raised to the **step count**, and the counter starts at 1. Skipping the bias correction is the most common Adam bug — it makes the first few steps roughly 3× too large.


In [ ]:
class MyAdam(Optimizer):
    """
    Adam optimizer.

    Args:
        params: iterable of parameters to optimize
        lr:     learning rate (default 1e-3)
        betas:  coefficients for the 1st- and 2nd-moment estimates
                (default (0.9, 0.999))
        eps:    term added to the denominator for numerical stability
                (default 1e-8)
    """

    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        if lr < 0.0:
            raise ValueError(f"Invalid learning rate: {lr}")
        if not 0.0 <= betas[0] < 1.0:
            raise ValueError(f"Invalid beta1: {betas[0]}")
        if not 0.0 <= betas[1] < 1.0:
            raise ValueError(f"Invalid beta2: {betas[1]}")
        defaults = dict(lr=lr, betas=betas, eps=eps)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        """
        Perform a single optimization step.

        Args:
            closure: optional callable that re-evaluates the model and returns
                     the loss
        Returns:
            the loss from the closure, if one was provided
        """
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr = group['lr']
            beta1, beta2 = group['betas']
            eps = group['eps']

            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad

                if grad.is_sparse:
                    raise RuntimeError("MyAdam does not support sparse gradients")

                state = self.state[p]

                # State initialization
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p)
                    state['exp_avg_sq'] = torch.zeros_like(p)

                exp_avg: torch.Tensor = state['exp_avg']
                exp_avg_sq: torch.Tensor = state['exp_avg_sq']
                state['step'] += 1

                # Biased moment estimates
                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)

                # Bias corrections
                bias_correction1 = 1 - beta1 ** state['step']
                bias_correction2 = 1 - beta2 ** state['step']

                # Update (formulated to match torch.optim.Adam exactly)
                denom = (exp_avg_sq.sqrt() / math.sqrt(bias_correction2)).add_(eps)
                step_size = lr / bias_correction1
                p.addcdiv_(exp_avg, denom, value=-step_size)

        return loss


## Part 2: AdamW

AdamW fixes a flaw in the usual way weight decay is bolted onto Adam. With **L2 regularization** the penalty is added to the *gradient* (`g ← g + w·θ`), so it passes through the adaptive denominator — weights with large gradients get regularized *less*. **AdamW decouples** the decay from the gradient entirely:

```
θ ← θ − lr·w·θ                     (directly on the parameter)
θ ← θ − lr·m̂_t / (√v̂_t + ε)        (the usual Adam step)
```

Note the decay is multiplied by `lr`, so the effective decay rate is `lr * weight_decay` — and a parameter with a **zero gradient must still decay**.


In [ ]:
class MyAdamW(Optimizer):
    """
    AdamW optimizer — Adam with decoupled weight decay.

    Args:
        params:       iterable of parameters to optimize
        lr:           learning rate (default 1e-3)
        betas:        coefficients for the moment estimates (default (0.9, 0.999))
        eps:          term for numerical stability (default 1e-8)
        weight_decay: weight decay coefficient (default 1e-2)
    """

    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8,
                 weight_decay=1e-2):
        if lr < 0.0:
            raise ValueError(f"Invalid learning rate: {lr}")
        if weight_decay < 0.0:
            raise ValueError(f"Invalid weight_decay: {weight_decay}")
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr = group['lr']
            beta1, beta2 = group['betas']
            eps = group['eps']
            weight_decay = group['weight_decay']

            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad

                if grad.is_sparse:
                    raise RuntimeError("MyAdamW does not support sparse gradients")

                state = self.state[p]

                # State initialization (same as Adam)
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p)
                    state['exp_avg_sq'] = torch.zeros_like(p)

                exp_avg: torch.Tensor = state['exp_avg']
                exp_avg_sq: torch.Tensor = state['exp_avg_sq']
                state['step'] += 1

                # Biased moment estimates and bias corrections
                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                bias_correction1 = 1 - beta1 ** state['step']
                bias_correction2 = 1 - beta2 ** state['step']
                denom = (exp_avg_sq.sqrt() / math.sqrt(bias_correction2)).add_(eps)
                step_size = lr / bias_correction1

                # Decoupled weight decay — directly on the parameter,
                # never through the gradient
                p.mul_(1 - lr * weight_decay)

                # Adam update
                p.addcdiv_(exp_avg, denom, value=-step_size)

        return loss


## Part 3: Muon

Muon ("Momentum Orthogonalized by Newton–Schulz") is a modern optimizer designed for training large transformers.

**Matrix parameters (`ndim >= 2`):**

1. Momentum: `buf ← μ·buf + g` — with Nesterov look-ahead, the update is `g + μ·buf`, otherwise just `buf`.
2. Orthogonalize the update with your `newton_schulz`.
3. Rescale by `√(max(m, n) / min(m, n))` — orthogonalization makes the update's magnitude depend on the matrix's aspect ratio, and this corrects for it.
4. Apply the same decoupled weight decay as AdamW, then `θ ← θ − lr·scale·update`.

**1D parameters** (biases, LayerNorm gains): orthogonalization is meaningless for a vector, so fall back to an **AdamW-style** update with its own betas and eps.

Typical LLM hyperparameters: `lr=0.02, momentum=0.95, nesterov=True, ns_steps=5, weight_decay=0.01`.


In [ ]:
class MyMuon(Optimizer):
    """
    Muon optimizer — Momentum with Newton–Schulz Orthogonalization.

    Args:
        params:       iterable of parameters to optimize
        lr:           learning rate (default 2e-2 — higher than Adam)
        momentum:     momentum coefficient μ (default 0.95)
        nesterov:     use Nesterov-style look-ahead momentum (default True)
        ns_steps:     Newton–Schulz iterations (default 5)
        weight_decay: decoupled weight decay (default 1e-2)
        adamw_betas:  betas for the AdamW fallback on 1D params (default (0.9, 0.95))
        adamw_eps:    epsilon for the AdamW fallback (default 1e-8)
    """

    def __init__(self, params, lr=2e-2, momentum=0.95, nesterov=True,
                 ns_steps=5, weight_decay=1e-2,
                 adamw_betas=(0.9, 0.95), adamw_eps=1e-8):
        defaults = dict(
            lr=lr, momentum=momentum, nesterov=nesterov,
            ns_steps=ns_steps, weight_decay=weight_decay,
            adamw_betas=adamw_betas, adamw_eps=adamw_eps,
        )
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr = group['lr']
            momentum = group['momentum']
            nesterov = group['nesterov']
            ns_steps = group['ns_steps']
            weight_decay = group['weight_decay']
            beta1, beta2 = group['adamw_betas']
            eps = group['adamw_eps']

            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad

                state = self.state[p]

                # State initialization
                if len(state) == 0:
                    state['step'] = 0
                    state['momentum_buffer'] = torch.zeros_like(p)
                    # AdamW state for the 1D fallback
                    state['exp_avg'] = torch.zeros_like(p)
                    state['exp_avg_sq'] = torch.zeros_like(p)

                state['step'] += 1
                buf: torch.Tensor = state['momentum_buffer']
                exp_avg: torch.Tensor = state['exp_avg']
                exp_avg_sq: torch.Tensor = state['exp_avg_sq']

                if p.ndim >= 2:
                    # ── Matrix parameter: Muon update ──

                    # Momentum buffer (Nesterov: look-ahead update)
                    buf.mul_(momentum).add_(grad)
                    if nesterov:
                        update = grad.add(buf, alpha=momentum)
                    else:
                        update = buf

                    # Orthogonalize the update
                    update = newton_schulz(update, steps=ns_steps)

                    # Aspect-ratio scale: orthogonalization makes the update's
                    # magnitude depend on the matrix shape; this corrects for it
                    m, n = update.shape
                    scale = math.sqrt(max(m, n) / min(m, n))

                    # Decoupled weight decay (same pattern as AdamW)
                    if weight_decay > 0:
                        p.mul_(1 - lr * weight_decay)

                    # Apply the orthogonalized update
                    p.add_(update, alpha=-lr * scale)

                else:
                    # ── 1D parameter (bias, norm gain): AdamW fallback ──

                    exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                    exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)

                    bias1 = 1 - beta1 ** state['step']
                    bias2 = 1 - beta2 ** state['step']
                    denom = (exp_avg_sq.sqrt() / math.sqrt(bias2)).add_(eps)
                    step_size = lr / bias1

                    if weight_decay > 0:
                        p.mul_(1 - lr * weight_decay)

                    p.addcdiv_(exp_avg, denom, value=-step_size)

        return loss


## Validation

Adam and AdamW are compared step-for-step against `torch.optim.Adam` / `torch.optim.AdamW` on a quadratic bowl — they should match to float precision. `newton_schulz` is checked for orthogonality, and Muon by training a tiny linear model.

Until your implementations are in place these tests will fail — that is expected.


In [ ]:
def test_adam():
    """Compare MyAdam with torch.optim.Adam on a quadratic objective."""
    print("Testing Adam...", end=" ")

    torch.manual_seed(42)
    D = 64
    target = torch.linspace(-1, 1, D)

    x_ref = torch.zeros(D, requires_grad=True)
    opt_ref = torch.optim.Adam([x_ref], lr=0.1, betas=(0.9, 0.999), eps=1e-8)

    x_my = torch.zeros(D, requires_grad=True)
    opt_my = MyAdam([x_my], lr=0.1, betas=(0.9, 0.999), eps=1e-8)

    for _ in range(100):
        loss_ref = ((x_ref - target) ** 2).mean()
        opt_ref.zero_grad()
        loss_ref.backward()
        opt_ref.step()

        loss_my = ((x_my - target) ** 2).mean()
        opt_my.zero_grad()
        loss_my.backward()
        opt_my.step()

    diff = (x_ref - x_my).abs().max().item()
    if diff < 1e-5:
        print(f"PASS  (max parameter diff: {diff:.2e})")
    else:
        print(f"FAIL  (max parameter diff: {diff:.2e})")


def test_adamw():
    """Compare MyAdamW with torch.optim.AdamW. Key: AdamW != Adam with wd."""
    print("Testing AdamW...", end=" ")

    torch.manual_seed(42)
    D = 64
    target = torch.linspace(-1, 1, D)

    x_ref = torch.zeros(D, requires_grad=True)
    opt_ref = torch.optim.AdamW([x_ref], lr=0.1, betas=(0.9, 0.999),
                                eps=1e-8, weight_decay=0.01)

    x_my = torch.zeros(D, requires_grad=True)
    opt_my = MyAdamW([x_my], lr=0.1, betas=(0.9, 0.999),
                     eps=1e-8, weight_decay=0.01)

    for _ in range(100):
        loss_ref = ((x_ref - target) ** 2).mean()
        opt_ref.zero_grad()
        loss_ref.backward()
        opt_ref.step()

        loss_my = ((x_my - target) ** 2).mean()
        opt_my.zero_grad()
        loss_my.backward()
        opt_my.step()

    diff = (x_ref - x_my).abs().max().item()
    if diff < 1e-5:
        print(f"PASS  (max parameter diff: {diff:.2e})")
    else:
        print(f"FAIL  (max parameter diff: {diff:.2e})")


def test_adamw_weight_decay():
    """AdamW's decoupled weight decay differs from Adam + L2 regularization."""
    print("Testing AdamW decoupled weight decay...", end=" ")

    torch.manual_seed(42)
    D = 16
    target = torch.randn(D)
    wd = 0.1

    x_adamw = torch.zeros(D, requires_grad=True)
    opt_adamw = torch.optim.AdamW([x_adamw], lr=0.1, weight_decay=wd)

    x_adam_l2 = torch.zeros(D, requires_grad=True)
    opt_adam_l2 = torch.optim.Adam([x_adam_l2], lr=0.1)

    for _ in range(50):
        loss_adamw = ((x_adamw - target) ** 2).mean()
        opt_adamw.zero_grad()
        loss_adamw.backward()
        opt_adamw.step()

        # Adam + L2: weight decay added to the gradient (NOT the same as AdamW)
        loss_l2 = ((x_adam_l2 - target) ** 2).mean()
        opt_adam_l2.zero_grad()
        loss_l2.backward()
        with torch.no_grad():
            x_adam_l2.grad.add_(wd * x_adam_l2)
        opt_adam_l2.step()

    diff = (x_adamw - x_adam_l2).abs().max().item()
    if diff > 1e-6:
        print(f"PASS  (AdamW != Adam+L2, max diff: {diff:.4f})")
    else:
        print("FAIL  (AdamW and Adam+L2 gave identical results)")


def test_newton_schulz():
    """Newton–Schulz should produce an approximately orthogonal matrix."""
    print("Testing newton_schulz...", end=" ")

    torch.manual_seed(42)
    m, n = 32, 64
    X = torch.randn(m, n)

    O = newton_schulz(X, steps=5)

    # For m <= n, O @ O.T should be close to I
    I_approx = O @ O.T
    diag_mean = I_approx.diag().mean().item()
    mask = ~torch.eye(m, dtype=torch.bool)
    offdiag_rms = I_approx[mask].square().mean().sqrt().item()

    if abs(diag_mean - 1.0) < 0.1 and offdiag_rms < 0.1:
        print(f"PASS  (diag mean: {diag_mean:.4f}, offdiag RMS: {offdiag_rms:.4f})")
    else:
        print(f"FAIL  (diag mean: {diag_mean:.4f}, offdiag RMS: {offdiag_rms:.4f})")


def test_muon():
    """Muon should reduce the loss on a small linear model."""
    print("Testing Muon...", end=" ")

    torch.manual_seed(42)
    B, D_in, D_out = 32, 16, 8

    model = nn.Linear(D_in, D_out)
    opt = MyMuon(model.parameters(), lr=0.02, momentum=0.95, weight_decay=1e-3)

    X = torch.randn(B, D_in)
    y = torch.randn(B, D_out)

    initial_loss = ((model(X) - y) ** 2).mean().item()

    for _ in range(200):
        loss = ((model(X) - y) ** 2).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()

    final_loss = ((model(X) - y) ** 2).mean().item()

    if final_loss < initial_loss * 0.5:
        print(f"PASS  (loss: {initial_loss:.4f} -> {final_loss:.4f})")
    else:
        print(f"FAIL  (loss: {initial_loss:.4f} -> {final_loss:.4f})")


test_newton_schulz()
test_adam()
test_adamw()
test_adamw_weight_decay()
test_muon()
